# 5 · Embed in your own website (`roadstyle.js`)

This is the decoupled path: **Python computes the styling once** (a JSON spec), and a small, reusable **`roadstyle.js`** draws it in the browser with your own UI on top. No Python at runtime, and — importantly — *you don't have to write the renderer*: it ships inside the package.

We'll generate a self-contained `output/web/` folder you can open or upload as-is.

In [ ]:
import pathlib
import geopandas as gpd
import roadstyle as rs

# A GeoDataFrame of road edges. roadstyle only needs a *road-class* column (default "highway")
# and line geometry in any CRS — it reprojects to web coordinates (EPSG:4326) for you.
edges = gpd.read_file(pathlib.Path("data") / "sundbyberg_edges.gpkg")
print(f"{len(edges):,} edges, CRS {edges.crs}")
edges[[c for c in edges.columns if c != edges.geometry.name]].head(3)


In [ ]:
import json, shutil, pathlib
from importlib.resources import files

web = pathlib.Path("output") / "web"
web.mkdir(parents=True, exist_ok=True)

# 1) Python bakes the styling into a JSON spec (no drawing, no HTML).
rs.save_spec(edges, web / "map_data.json", theme="dark", palette="highsat")

# 2) Copy the canonical renderer + its CSS straight out of the installed package.
for asset in ("roadstyle.js", "roadstyle.css"):
    shutil.copy(files("roadstyle") / "static" / asset, web / asset)

print("wrote", *(p.name for p in web.iterdir()))

### Tune interactions without touching code
`roadstyle.js` reads an optional `interaction_config.json` — hover colour, selection glow, map toggles. A designer can edit this file; the JS never changes.

In [ ]:
(web / "interaction_config.json").write_text(json.dumps({
    "hoverColor": "#38bdf8",
    "hoverExtraWidth": 3,
    "selectionStyle": {
        "glow":   {"color": "#38bdf8", "width": 14, "opacity": 0.5},
        "casing": {"color": "#0f172a", "width": 8,  "opacity": 1.0},
        "core":   {"color": "#ffffff", "width": 4,  "opacity": 1.0}
    },
    "doubleClickZoom": True,
    "zoomControl": True
}, indent=2))

### A page that wires a custom sidebar to the map
The renderer is headless: it exposes `getRoadClasses()` and `setFilter()`. Here we build our own checkbox sidebar in plain HTML/CSS and connect it — the pattern for embedding in any site.

In [ ]:
INDEX_HTML = """<!DOCTYPE html>
<html lang="en"><head><meta charset="utf-8"><title>Road network</title>
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css"/>
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<link rel="stylesheet" href="./roadstyle.css"/>
<style>
  body{margin:0;display:flex;height:100vh;font-family:system-ui,sans-serif;background:#0f172a;color:#e2e8f0}
  #sidebar{width:240px;padding:18px;box-sizing:border-box;border-right:1px solid #1e293b;overflow:auto}
  #sidebar h1{font-size:16px;margin:0 0 4px;color:#38bdf8}
  #sidebar p{font-size:12px;color:#94a3b8;margin:0 0 14px}
  label{display:flex;gap:8px;align-items:center;font-size:13px;padding:4px 6px;border-radius:6px}
  label:hover{background:#1e293b}
  #map{flex:1}
</style></head>
<body>
  <div id="sidebar">
    <h1>Road network</h1>
    <p>Styling precomputed in Python; filtered live in the browser.</p>
    <div id="filters"></div>
  </div>
  <div id="map"></div>
  <script src="./roadstyle.js"></script>
  <script>
    const m = new RoadStyleMap("map", { widgets: { legend: true } });
    m.load("./map_data.json", "./interaction_config.json").then(() => {
      const box = document.getElementById("filters");
      m.getRoadClasses().forEach(cls => {
        const l = document.createElement("label");
        l.innerHTML = `<input type="checkbox" value="${cls}" checked> ${cls}`;
        box.appendChild(l);
      });
      box.addEventListener("change", () => {
        const on = [...box.querySelectorAll("input:checked")].map(c => c.value);
        m.setFilter(on);
      });
    });
  </script>
</body></html>
"""
(web / "index.html").write_text(INDEX_HTML)
print("Open", (web / "index.html").resolve(), "in a browser — the folder is fully self-contained.")

### Zero-JavaScript alternative
If you don't need a custom UI at all, `save()` / `to_iframe()` give you a finished map with no frontend work (notebook 4). Reach for `roadstyle.js` only when you want your *own* page around the map.